In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf

In [ ]:
# Constants
RISK_FREE_RATE = 0.042

INDICES = ["SPY", "IWM", "DIA"]
TICKERS = ["MU", "EQIX", "ETN", "AMD", "EL", "WDC", "ANET"]

PERIOD = "10y"

In [ ]:
# Download ticker data
yfinance_data = yf.download(tickers=TICKERS+INDICES, period=PERIOD)['Close']

In [ ]:
def calculate_returns(prices: pd.DataFrame) -> pd.DataFrame:
    daily_rets = prices.pct_change().dropna()
    return daily_rets

def calculate_vol(returns: pd.DataFrame) -> float:
    annualized_vol = np.sqrt(252) * returns.std()
    return annualized_vol

def calculate_beta(x: pd.Series, y: pd.Series):
    aligned = pd.concat([y, x], axis=1).dropna()
    if aligned.shape[0] < 2:
        return np.nan
    cov = np.cov(aligned.iloc[:,0], aligned.iloc[:,1], ddof=1)[0,1]
    var = np.var(aligned.iloc[:,1], ddof=1)
    return cov / var if var != 0 else np.nan


def weekly_drawdowns(prices: pd.DataFrame, weeks: int = 52):
    week = prices.resample('W').agg(['max', 'min']).dropna()
    week['drawdown'] = (week['min'] - week['max']) / week['max']
    last = week.tail(weeks)
    if last.empty:
        return np.nan, np.nan
    avg = last['drawdown'].mean()
    max = last['drawdown'].min()
    return avg, max

def total_return(prices: pd.DataFrame, years: int = 10):
    end_date = prices.dropna().index.max()
    start_date = end_date - pd.DateOffset(years=years)
    window = prices.loc[start_date:end_date].dropna()
    if window.empty or window.shape[0] < 2:
        return np.nan, np.nan
    tr = window.iloc[-1] / window.iloc[0] - 1.0
    ar = (1.0 + tr) ** (1/years) - 1.0
    return tr, ar

def risk_free_daily():
    rf = yf.download("^IRX", period=PERIOD)['Close'].dropna()
    rf = rf / 100
    rf_daily = (1 + rf) ** 1/252 - 1
    return rf_daily

def sharpe_ratio(returns: pd.Series, rf_daily: pd.Series):
    aligned = pd.concat([returns, rf_daily], axis=1).fillna(method="ffill")
    aligned.columns = ['ret', 'rf']
    aligned = aligned.dropna()
    if aligned.empty or aligned['ret'].std() == 0:
        return np.nan
    ex = aligned['ret'] - aligned['rf']
    return ex.mean() / aligned['ret'].std() * np.sqrt(252)

def tracking_error(portfolio: pd.Series, benchmark: pd.Series):
    aligned = pd.concat([portfolio, benchmark], axis=1).dropna()
    if aligned.empty:
        return np.nan
    active = aligned.iloc[:,0] - aligned.iloc[:,1]
    return active.std() * np.sqrt(252)

In [ ]:
prices = yfinance_data
returns = calculate_returns(yfinance_data)

asset_prices = prices[TICKERS]
asset_returns = returns[TICKERS]
etf_prices = prices[INDICES]
etf_returns = returns[INDICES]

In [ ]:
w = np.repeat(1/len(TICKERS), len(TICKERS))
portfolio_returns = (asset_returns * w).sum(axis=1)
portfolio_prices = (1 + portfolio_returns).cumprod() * 100

In [ ]:
ret_3m = asset_returns.last('63D')
ret_12m = asset_returns.last('252D')
spy_12m = etf_returns['SPY'].last('252D')
iwm_12m = etf_returns['IWM'].last('252D')
dia_12m = etf_returns['DIA'].last('252D')

rows = []
for ticker in TICKERS:
    weight = 1/len(TICKERS)
    vol_3m = calculate_vol(ret_3m[ticker].dropna())
    beta_spy = calculate_beta(ret_12m[ticker], spy_12m)
    beta_iwm = calculate_beta(ret_12m[ticker], iwm_12m)
    beta_dia = calculate_beta(ret_12m[ticker], dia_12m)
    avg_dd, max_dd = weekly_drawdowns(asset_prices[ticker])
    tr10, ar10 = total_return(asset_prices[ticker])
    rows.append([ticker, weight, vol_3m, beta_spy, beta_iwm, beta_dia, avg_dd, max_dd, tr10. ar10])

asset_factors = pd.DataFrame(rows, columns=[
    'Ticker', 'Portfolio Weight', 'Annualized Volatility (3M)', 'SPY Beta', 'IWM Beta', 'DIA Beta',
    'Avg Weekly Drawdown', 'Max Weekly Drawdown', 'Total Return (10Y)', 'Annualized Total Return'
])

# print or save to csv here

In [ ]:
rf_daily = risk_free_daily()
portfolio_sharpe = sharpe_ratio(portfolio_returns, rf_daily)
portfolio_volatility = calculate_vol(portfolio_returns.dropna())

rows = []
for etf in INDICES:
    etf_return = etf_returns[etf]
    correlation = portfolio_returns.corr(etf_return)
    aligned = pd.concat([portfolio_returns, etf_return], axis=1).dropna()
    cov = np.cov(aligned.T)[0, 1] if not aligned.empty else np.nan
    tracking_error = tracking_error(portfolio_returns, etf_return)
    etf_volatility = calculate_vol(etf_return.dropna())
    vol_spread = portfolio_volatility - etf_volatility
    rows.append([etf, correlation, cov, tracking_error, portfolio_sharpe, vol_spread])

portfolio_vs_etf = pd.DataFrame(rows, columns=[
    'ETF', 'Correlation', 'Covariance', 'Tracking Error', 'Sharpe Ratio', 'Annual Volatility Spread'
])

# print or save to csv here

In [ ]:
corr_df = pd.concat([portfolio_returns.rename('Portfolio'), etf_returns, asset_returns], axis=1).dropna().corr()